# Phase II foundation: time-aligned liquidity prediction
**AMS 520 / AMS 518 — research working notebook, not a completed empirical submission.**

This notebook compares persistence with residual ridge models on the *same eligible clock observations*. It builds a 30-second-ahead displayed-cost target, splits whole UTC receipt dates, and learns scaling and coefficients using training dates only. Validation chooses regularization; test dates are untouched until scoring. The default is **SYNTHETIC integration data**, not real market observations or evidence that a hedge improves.

The theoretical report is `reports/theory_literature.tex` (compiled PDF supplied separately). For source limitations and references see `DATA_CARD.md`, `MODEL_ASSUMPTIONS.md`, `AI_USAGE.md` and `docs/liquidity-learning.md`.

Run the source notebook without private outputs in Git. For real data, copy it outside the repository and set `DTS_REPO_ROOT` plus `DTS_LIQUIDITY_MANIFEST` before starting the kernel. The manifest must specify stopped exports and **predeclared whole-date splits**. No token, broker session or database is read by this notebook.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root_hint = os.environ.get("DTS_REPO_ROOT")
candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/"research/liquidity_aware_hedging/liquidity_dataset.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set DTS_REPO_ROOT to the existing repository checkout")
sys.path[:0] = [str(ROOT/"tools"), str(ROOT/"research/liquidity_aware_hedging")]
from liquidity_study import prepare_inputs
from liquidity_dataset import build_dataset, FEATURE_GROUPS
from liquidity_baselines import split_by_date, run_baselines
manifest = os.environ.get("DTS_LIQUIDITY_MANIFEST") or None
payloads, configuration, partitions = prepare_inputs(synthetic=not bool(manifest), manifest=manifest)
print("OBSERVED export analysis" if manifest else "SYNTHETIC integration fixture — not empirical market evidence")
print(configuration)

## 1. Information clock and eligibility
At clock time $t$, apply only callbacks received at or before $t$. A target at $t+H$ uses the last observed reconstructed state **at or before** that target time, subject to a side-update age bound. Never choose the nearest future update.

Reset/invalid states break continuity even between clock ticks. The feature window and target horizon must remain within one uninterrupted segment and UTC date. Missing capacity is not replaced by a price. This produces a **conditional, selected sample**: future outages or insufficient displayed size can remove labels. Coverage losses must be reported, not hidden.

In [ ]:
examples, data_manifest = build_dataset(payloads, configuration)
audit = pd.DataFrame([{"source_sha256": s["source_sha256"], **s["eligibility"]} for s in data_manifest["sessions"]]).fillna(0)
display(audit)
print("Eligible rows:", len(examples))
print("Dataset fingerprint:", data_manifest["dataset_sha256"])
assert (examples.feature_observation_ns <= examples.decision_ns).all()
assert (examples.label_observation_ns <= examples.label_ns).all()
assert (examples.label_ns-examples.decision_ns == configuration.horizon_seconds*10**9).all()
display(examples[["date","decision_us","label_us","current_target_bps","target_bps"]].head())

## 2. Date-blocked evaluation
One captured session is insufficient for a train/validation/test claim. All captures on one UTC date stay in one partition. An explicit check ensures labels in an earlier partition end before the next partition's feature windows begin. This excludes cross-split overlapping labels; it does **not** make errors within a date independent.

All feature families include the observable current target and trailing price diagnostics. `top` adds L1 state and OFI; `depth` adds total recorded size and imbalance. Therefore `history` is not a strictly price-only baseline. Quantity, horizon and age rules must be fixed before looking at the test results.

In [ ]:
kwargs = {k+"_dates":v for k,v in partitions.items()} if partitions else {}
indices, date_split = split_by_date(examples, **kwargs)
display(pd.DataFrame([{"partition": k, "dates": ", ".join(date_split[k]), "rows":len(v)} for k,v in indices.items()]))
print(FEATURE_GROUPS)

## 3. Persistence and residual ridge
Persistence forecasts the current displayed cost. Ridge learns a correction:
$$\hat y_{t+H}=y_t+b+z_t^\top\beta.$$
The fitted objective is mean squared residual loss plus $\lambda\|\beta\|_2^2$. The intercept is not penalized. Training-only standardization and an SVD solve avoid explicit matrix inversion. Negative predictions are counted, not silently clipped.

Hyperparameters minimize equal-date validation MSE. Models are **not refitted** on validation or test data in this first implementation. All test feature-set results are predeclared ablations, not an invitation to select another test winner.

In [ ]:
report, predictions = run_baselines(examples, data_manifest, partitions)
summary = pd.DataFrame([{ "model": name, "test_MAE_bps": score["mae_bps"],
                         "test_RMSE_bps": score["rmse_bps"],
                         "equal_date_MAE_bps": score["macro_day_mae_bps"],
                         "negative_prediction_fraction": score["negative_prediction_fraction"]}
                        for name, score in report["test"].items()])
display(summary)
print("Selected on validation only:", report["selected_on_validation"])
print("All figures below:", data_manifest["source"])
display(pd.DataFrame(report["paired_day_differences"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary.model, summary.equal_date_MAE_bps)
ax.set_ylabel("Equal-date MAE (basis points)")
ax.set_title(data_manifest["source"] + " — held-out date diagnostics")
fig.tight_layout()
plt.show()

In [ ]:
one_day = predictions[predictions.date == predictions.date.iloc[0]].head(180)
fig, ax = plt.subplots(figsize=(10, 4))
for name in dict.fromkeys(("target_bps", "persistence", report["selected_on_validation"])):
    ax.plot(np.arange(len(one_day)), one_day[name], label=name)
ax.set_xlabel("Held-out eligible clock observation (gaps are not equal elapsed time)")
ax.set_ylabel("Displayed cost / spread (basis points)")
ax.set_title(data_manifest["source"] + " — observed target vs forecasts")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Interpretation and next experiments
The synthetic fixture deliberately has structured dynamics. A lower error here establishes only that the pipeline executes and can recover structure in that fixture. It does not validate a real feed, predict actual fills, reduce real CVaR, or calibrate LSV.

Before an empirical claim, record multiple sessions; freeze the manifest/split; inspect conditional sample retention by day and market state; compare held-out persistence and ridge; and reserve additional dates for replication. Overlapping errors require dependence-aware uncertainty, not IID standard errors. A future availability classifier should address missing future depth rather than treating unpriced observations as zero cost.

The AMS 518 extension needs a separately validated cost/scenario interface and CVaR optimizer. A point forecast of mean cost is not a conditional tail distribution. The LSV pricing measure is distinct from real-world risk scenarios. Neither extension is implemented by this notebook.

AI assistance includes the implementation, tests, notebook, derivations and review draft. Independently verify the mathematics and sources, obtain required cross-course approval, and maintain the human-contribution log.

In [ ]:
# Rerun determinism excludes no model-dependent result: the entire pure report matches.
again, again_predictions = run_baselines(examples, data_manifest, partitions)
assert report == again
pd.testing.assert_frame_equal(predictions, again_predictions)
print("PASS: repeatable features, date partitions, model selection and predictions")
print("No automatic files, credentials, broker calls or database writes were made.")